In [ ]:
# Public-repository path setup.
# Run from anywhere inside the repository, or set CEFTAZIDIME_PROJECT_ROOT.
import os
from pathlib import Path

def _repo_root():
    env = os.environ.get("CEFTAZIDIME_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "README.md").exists() and (candidate / "03_Notebooks").exists():
            return candidate
    return here

def _previous_project_root(project_root):
    env = os.environ.get("GENOME_MIC_AMR_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    return (project_root / "external" / "Genome_MIC_AMR_Emergence").resolve()

PROJECT_ROOT = _repo_root()

#@title Cell 25.1 - Overview, paths, and final analysis design
# Purpose:
# Final focused analysis before manuscript writing.
# Compare the 16 high-MIC pathogens with their 3 matched comparators across
# the 14 supported mapped chromosomal subwindows.
#
# Outputs:
# 25A: 16 high-MIC pathogens x 14 loci, mean unitig dissimilarity to 3 comparators
# 25B: exact high-MIC regional state absent/present among the 3 comparators
# 25C: all 48 matched pairs x 14 loci, pairwise unitig dissimilarity
# 25D: chromosome-position summary of the supported locus distribution
#
# Important: the 14 loci were selected previously using MIC-guided ablation.
# This is secondary descriptive interpretation, not independent confirmation.

from pathlib import Path
import json, re
import numpy as np
import pandas as pd
from scipy import sparse
import matplotlib.pyplot as plt
from IPython.display import display
PROJECT_ROOT = _repo_root()
RESULTS_TABLE_DIR=PROJECT_ROOT/'05_Results'/'Tables'
RESULTS_FIGURE_DIR=PROJECT_ROOT/'05_Results'/'Figures'
RESULTS_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
UNITIG_DIR=PROJECT_ROOT/'04_Intermediate'/'10_Whole_Chromosome_Unitigs'
UNITIG_MATRIX=UNITIG_DIR/'10_variable_unitig_matrix_176xM.npz'
UNITIG_SAMPLES=UNITIG_DIR/'10_unitig_sample_order.csv'
NB21_DIR=PROJECT_ROOT/'04_Intermediate'/'21_Mapped_Priority_Window_Subwindow_Ablation'
NB21_ASSIGNMENT=NB21_DIR/'21_mapped_subwindow_assignment.npz'
NB21_RESULTS=RESULTS_TABLE_DIR/'21_mapped_subwindow_matched_random_results.csv'
NB21_QC=RESULTS_TABLE_DIR/'21_mapped_subwindow_final_QC.csv'
NB24_DIR=PROJECT_ROOT/'04_Intermediate'/'24_Exact_Mapped_Sequence_State_Visualization'
NB24_STATES=NB24_DIR/'24_exact_regional_state_matrix.npz'
NB24_QC=RESULTS_TABLE_DIR/'24_exact_mapped_sequence_state_visualization_final_QC.csv'
NB25_DIR=PROJECT_ROOT/'04_Intermediate'/'25_High_MIC_Matched_Comparator_Locus_Distribution'
NB25_DIR.mkdir(parents=True, exist_ok=True)
PAIR_RESULTS=RESULTS_TABLE_DIR/'25_48_matched_pair_locus_dissimilarity.csv.gz'
HIGH_RESULTS=RESULTS_TABLE_DIR/'25_16_high_MIC_locus_distribution_summary.csv'
LOCUS_SUMMARY=RESULTS_TABLE_DIR/'25_supported_locus_distribution_summary.csv'
FIG25A=RESULTS_FIGURE_DIR/'25A_high_MIC_mean_matched_comparator_locus_dissimilarity.png'
FIG25B=RESULTS_FIGURE_DIR/'25B_high_MIC_exact_state_absent_from_matched_comparators.png'
FIG25C=RESULTS_FIGURE_DIR/'25C_48_matched_pairs_supported_locus_dissimilarity.png'
FIG25D=RESULTS_FIGURE_DIR/'25D_supported_locus_distribution_summary.png'
FINAL_QC=RESULTS_TABLE_DIR/'25_high_MIC_matched_comparator_locus_distribution_final_QC.csv'
COMPLETE=NB25_DIR/'25_HIGH_MIC_MATCHED_COMPARATOR_LOCUS_DISTRIBUTION_COMPLETE.json'
EXPECTED_PATHOGENS=176
EXPECTED_UNITIGS=1_287_844
EXPECTED_HIGH=16
EXPECTED_PAIRS=48
EXPECTED_COMPARATORS=23
EXPECTED_REGIONS=14
for p in [UNITIG_MATRIX,UNITIG_SAMPLES,NB21_ASSIGNMENT,NB21_RESULTS,NB21_QC,NB24_STATES,NB24_QC]:
    assert p.exists(), f'Required input not found: {p}'
matching=list(PROJECT_ROOT.rglob('03_matched_pairs_16x3.csv'))
assert len(matching)==1, f'Expected exactly one 03_matched_pairs_16x3.csv; found {matching}'
MATCHED_PAIRS_FILE=matching[0]
print('Notebook 25 - High-MIC Matched-Comparator Locus Distribution')
print('Matched-pair file:', MATCHED_PAIRS_FILE)
print('Transition: Cell 25.2 will verify inputs and resolve the 48 matched pairs.')


In [ ]:
#@title Cell 25.2 - Verify inputs and resolve the 48 matched pairs
# Purpose: load the fixed cohort, supported loci, Notebook 24 states, and matching file.

assert bool(pd.read_csv(NB21_QC).loc[0,'final_QC_pass'])
assert bool(pd.read_csv(NB24_QC).loc[0,'final_QC_pass'])
samples=pd.read_csv(UNITIG_SAMPLES).sort_values('sample_index').reset_index(drop=True)
assert len(samples)==EXPECTED_PATHOGENS
assert np.array_equal(samples['sample_index'].to_numpy(int),np.arange(EXPECTED_PATHOGENS))
y=samples['log2_mic'].to_numpy(float)
high_from_mic=np.flatnonzero(y>2.0)
assert len(high_from_mic)==EXPECTED_HIGH
X=sparse.load_npz(UNITIG_MATRIX).tocsc()
assert X.shape==(EXPECTED_PATHOGENS,EXPECTED_UNITIGS)

mapped=pd.read_csv(NB21_RESULTS)
mapped=mapped.loc[mapped['larger_drop_than_matched_random_after_within_parent_BH'].astype(bool)].copy()
assert len(mapped)==EXPECTED_REGIONS
mapped['reference_midpoint_Mb']=(mapped['reference_start_0_based']+mapped['reference_end_0_based_exclusive'])/2e6
mapped=mapped.sort_values('reference_midpoint_Mb').reset_index(drop=True)
with np.load(NB21_ASSIGNMENT) as z:
    subwindow_code=np.asarray(z['subwindow_code'],dtype=np.int16)
with np.load(NB24_STATES) as z:
    state_matrix=np.asarray(z['state_matrix'],dtype=np.int16)
    state_names=np.asarray(z['region_names']).astype(str)
assert state_matrix.shape==(EXPECTED_PATHOGENS,EXPECTED_REGIONS)
assert np.array_equal(state_names,mapped['subwindow_name'].astype(str).to_numpy())

pairs_raw=pd.read_csv(MATCHED_PAIRS_FILE)
print('Matched-pair columns:', list(pairs_raw.columns))

def norm(s): return re.sub(r'[^a-z0-9]+','_',str(s).lower()).strip('_')
name_map={col:norm(col) for col in pairs_raw.columns}
high_cols=[c for c,n in name_map.items() if 'high' in n and any(t in n for t in ['sample','biosample','pathogen','index','id'])]
comp_cols=[c for c,n in name_map.items() if ('comparator' in n or 'matched' in n) and any(t in n for t in ['sample','biosample','pathogen','index','id'])]
assert len(high_cols)==1 and len(comp_cols)==1, f'Could not uniquely resolve pair columns. high={high_cols}; comparator={comp_cols}'

def resolve(series):
    num=pd.to_numeric(series,errors='coerce')
    if num.notna().all() and np.allclose(num,num.astype(int)):
        vals=num.astype(int)
        if set(vals).issubset(set(samples['sample_index'].astype(int))): return vals.to_numpy(int),'sample_index'
    vals=series.astype(str).str.strip()
    matches=[]
    for col in samples.columns:
        if col in ['sample_index','log2_mic']: continue
        s=samples[col].astype(str).str.strip()
        if s.duplicated().any(): continue
        mapping=dict(zip(s,samples['sample_index'].astype(int)))
        if vals.isin(mapping.keys()).all(): matches.append((col,mapping))
    assert len(matches)==1, f'Identifier mapping ambiguous for {series.name}: {[m[0] for m in matches]}'
    col,mapping=matches[0]
    return vals.map(mapping).to_numpy(int),col

hi,hi_source=resolve(pairs_raw[high_cols[0]])
co,co_source=resolve(pairs_raw[comp_cols[0]])
pairs=pairs_raw.copy()
pairs['high_sample_index']=hi
pairs['comparator_sample_index']=co
pairs['high_log2_mic']=y[hi]
pairs['comparator_log2_mic']=y[co]
pairs['log2_mic_difference']=pairs['high_log2_mic']-pairs['comparator_log2_mic']
assert len(pairs)==EXPECTED_PAIRS
assert pairs['high_sample_index'].nunique()==EXPECTED_HIGH
assert pairs['comparator_sample_index'].nunique()==EXPECTED_COMPARATORS
assert (pairs.groupby('high_sample_index').size()==3).all()
assert set(pairs['high_sample_index'])==set(high_from_mic)
assert (pairs['log2_mic_difference']>=2).all()
print('Input QC: PASS')
print('High identifier source:',hi_source)
print('Comparator identifier source:',co_source)
print('48 pairs / 16 high-MIC / 23 unique matched comparators confirmed.')
print('Transition: Cell 25.3 will calculate regional differences for all 48 pairs.')


In [ ]:
#@title Cell 25.3 - Calculate regional differences for all 48 matched pairs
# Purpose: exact-state agreement and Jaccard dissimilarity at all 14 loci.

region_unitigs=[]
for _,row in mapped.iterrows():
    idx=np.flatnonzero(subwindow_code==int(row['global_subwindow_code']))
    assert len(idx)==int(row['n_unitigs'])
    region_unitigs.append(idx)

pair_matrix=np.zeros((EXPECTED_PAIRS,EXPECTED_REGIONS),float)
exact_diff=np.zeros((EXPECTED_PAIRS,EXPECTED_REGIONS),np.uint8)
rows=[]
for pr,pair in pairs.reset_index(drop=True).iterrows():
    hi=int(pair['high_sample_index']); co=int(pair['comparator_sample_index'])
    for r,row in mapped.iterrows():
        idx=region_unitigs[r]
        a=X[hi,idx]; b=X[co,idx]
        na=float(a.sum()); nb=float(b.sum()); inter=float(a.multiply(b).sum())
        union=na+nb-inter
        sim=inter/union if union>0 else 1.0
        dis=1.0-sim
        same=bool(state_matrix[hi,r]==state_matrix[co,r])
        pair_matrix[pr,r]=dis; exact_diff[pr,r]=int(not same)
        rows.append({'pair_row_index':pr,'high_sample_index':hi,'comparator_sample_index':co,
                     'high_log2_mic':float(y[hi]),'comparator_log2_mic':float(y[co]),
                     'subwindow_name':str(row['subwindow_name']),'reference_midpoint_Mb':float(row['reference_midpoint_Mb']),
                     'exact_state_same':same,'jaccard_similarity':sim,'jaccard_dissimilarity':dis})
pair_results=pd.DataFrame(rows)
assert len(pair_results)==EXPECTED_PAIRS*EXPECTED_REGIONS
pair_results.to_csv(PAIR_RESULTS,index=False,compression='gzip')
np.savez_compressed(NB25_DIR/'25_pair_matrices.npz',pair_dissimilarity=pair_matrix,exact_difference=exact_diff)
print('Matched-pair calculation: PASS')
print('Pair-region rows:',len(pair_results))
print('Transition: Cell 25.4 will aggregate the 3 comparators for each high-MIC pathogen.')


In [ ]:
#@title Cell 25.4 - Aggregate the three comparators for each high-MIC pathogen
# Purpose: build 16 x 14 matrices and the final locus summary.

high_table=(pairs[['high_sample_index','high_log2_mic']].drop_duplicates()
            .sort_values(['high_log2_mic','high_sample_index'],ascending=[False,True]).reset_index(drop=True))
high_order=high_table['high_sample_index'].to_numpy(int)
high_mean=np.zeros((EXPECTED_HIGH,EXPECTED_REGIONS),float)
high_absent=np.zeros((EXPECTED_HIGH,EXPECTED_REGIONS),np.uint8)
high_rows=[]
pr_pairs=pairs.reset_index(drop=True)
for hrow,hi in enumerate(high_order):
    pair_rows=pr_pairs.index[pr_pairs['high_sample_index']==hi].to_numpy(int)
    comps=pr_pairs.loc[pair_rows,'comparator_sample_index'].to_numpy(int)
    assert len(pair_rows)==3
    for r,row in mapped.iterrows():
        d=pair_matrix[pair_rows,r]
        hs=int(state_matrix[hi,r]); cs=state_matrix[comps,r]
        absent=bool(np.all(cs!=hs))
        high_mean[hrow,r]=d.mean(); high_absent[hrow,r]=int(absent)
        high_rows.append({'high_sample_index':int(hi),'high_log2_mic':float(y[hi]),
                          'subwindow_name':str(row['subwindow_name']),'reference_midpoint_Mb':float(row['reference_midpoint_Mb']),
                          'mean_jaccard_dissimilarity_to_3_matched_comparators':float(d.mean()),
                          'minimum_jaccard_dissimilarity_to_3_matched_comparators':float(d.min()),
                          'maximum_jaccard_dissimilarity_to_3_matched_comparators':float(d.max()),
                          'high_exact_state_absent_from_all_3_matched_comparators':absent})
high_results=pd.DataFrame(high_rows)
high_results.to_csv(HIGH_RESULTS,index=False)

locus_rows=[]
for r,row in mapped.iterrows():
    locus_rows.append({'subwindow_name':str(row['subwindow_name']),'reference_midpoint_Mb':float(row['reference_midpoint_Mb']),
                       'mean_jaccard_dissimilarity_across_48_matched_pairs':float(pair_matrix[:,r].mean()),
                       'median_jaccard_dissimilarity_across_48_matched_pairs':float(np.median(pair_matrix[:,r])),
                       'number_of_16_high_MIC_pathogens_with_exact_state_absent_from_all_3_comparators':int(high_absent[:,r].sum()),
                       'fraction_of_16_high_MIC_pathogens_with_exact_state_absent_from_all_3_comparators':float(high_absent[:,r].mean()),
                       'Notebook21_observed_drop_after_removal':float(row['observed_drop_after_removal']),
                       'Notebook21_within_parent_BH_q_value':float(row['within_parent_BH_q_value'])})
locus_summary=pd.DataFrame(locus_rows)
locus_summary.to_csv(LOCUS_SUMMARY,index=False)
np.savez_compressed(NB25_DIR/'25_high_MIC_locus_matrices.npz',high_mean_dissimilarity=high_mean,
                    high_exact_state_absent=high_absent,high_sample_index=high_order,
                    high_log2_mic=y[high_order],region_names=mapped['subwindow_name'].astype(str).to_numpy(dtype='U16'))
print('Aggregation QC: PASS')
display(locus_summary)
print('Transition: Cell 25.5 will draw the main 16 x 14 dissimilarity heatmap.')


In [ ]:
#@title Cell 25.5 - Draw 16 high-MIC pathogen x locus dissimilarity heatmap
# Rows: high-MIC pathogens ordered from highest to lowest MIC.
# Columns: 14 supported loci in chromosome order.
# Cell: mean Jaccard dissimilarity to the 3 matched comparators.

fig,ax=plt.subplots(figsize=(13,7))
im=ax.imshow(high_mean,aspect='auto',interpolation='nearest',vmin=0,vmax=1)
ax.set_title('Regional sequence differences between 16 high-MIC pathogens and matched comparators')
ax.set_xlabel('Supported mapped chromosomal subwindow'); ax.set_ylabel('High-MIC pathogen')
ax.set_xticks(np.arange(EXPECTED_REGIONS)); ax.set_xticklabels(mapped['subwindow_name'].astype(str),rotation=90)
ax.set_yticks(np.arange(EXPECTED_HIGH)); ax.set_yticklabels([f'H{i+1:02d}  MIC={y[h]:g} log2' for i,h in enumerate(high_order)])
cb=fig.colorbar(im,ax=ax); cb.set_label('Mean Jaccard dissimilarity to 3 matched comparators')
fig.tight_layout(); fig.savefig(FIG25A,dpi=300,bbox_inches='tight'); fig.savefig(FIG25A.with_suffix('.pdf'),bbox_inches='tight'); plt.show(); plt.close(fig)
print('Saved:',FIG25A)
print('Transition: Cell 25.6 will show exact regional states absent from all 3 matched comparators.')


In [ ]:
#@title Cell 25.6 - Draw exact-state absence heatmap
# Cell = 1 when the high-MIC pathogen's exact regional state is absent from all
# three matched comparators; 0 when at least one comparator has the same state.

fig,ax=plt.subplots(figsize=(13,7))
im=ax.imshow(high_absent,aspect='auto',interpolation='nearest',vmin=0,vmax=1)
ax.set_title('Exact regional states of high-MIC pathogens relative to matched comparators')
ax.set_xlabel('Supported mapped chromosomal subwindow'); ax.set_ylabel('High-MIC pathogen')
ax.set_xticks(np.arange(EXPECTED_REGIONS)); ax.set_xticklabels(mapped['subwindow_name'].astype(str),rotation=90)
ax.set_yticks(np.arange(EXPECTED_HIGH)); ax.set_yticklabels([f'H{i+1:02d}  MIC={y[h]:g} log2' for i,h in enumerate(high_order)])
cb=fig.colorbar(im,ax=ax,ticks=[0,1]); cb.set_ticklabels(['State seen in ≥1 comparator','State absent from all 3 comparators'])
fig.tight_layout(); fig.savefig(FIG25B,dpi=300,bbox_inches='tight'); fig.savefig(FIG25B.with_suffix('.pdf'),bbox_inches='tight'); plt.show(); plt.close(fig)
print('Saved:',FIG25B)
print('Transition: Cell 25.7 will show all 48 matched pairs individually.')


In [ ]:
#@title Cell 25.7 - Draw all 48 matched-pair locus-dissimilarity profiles
# Rows are grouped by high-MIC pathogen; columns are the 14 supported loci.

pr=pairs.reset_index(drop=True)
order=[]
for hi in high_order:
    order.extend(pr.index[pr['high_sample_index']==hi].tolist())
order=np.asarray(order,int)
ordered=pair_matrix[order,:]
fig,ax=plt.subplots(figsize=(13,10))
im=ax.imshow(ordered,aspect='auto',interpolation='nearest',vmin=0,vmax=1)
ax.set_title('Regional unitig dissimilarity across all 48 matched pairs')
ax.set_xlabel('Supported mapped chromosomal subwindow'); ax.set_ylabel('Matched pairs grouped by high-MIC pathogen')
ax.set_xticks(np.arange(EXPECTED_REGIONS)); ax.set_xticklabels(mapped['subwindow_name'].astype(str),rotation=90)
centres=[1+3*i for i in range(EXPECTED_HIGH)]
ax.set_yticks(centres); ax.set_yticklabels([f'H{i+1:02d}' for i in range(EXPECTED_HIGH)])
for i in range(1,EXPECTED_HIGH): ax.axhline(3*i-0.5,linewidth=0.5,alpha=0.25)
cb=fig.colorbar(im,ax=ax); cb.set_label('Jaccard dissimilarity')
fig.tight_layout(); fig.savefig(FIG25C,dpi=300,bbox_inches='tight'); fig.savefig(FIG25C.with_suffix('.pdf'),bbox_inches='tight'); plt.show(); plt.close(fig)
print('Saved:',FIG25C)
print('Transition: Cell 25.8 will draw the chromosome-position summary.')


In [ ]:
#@title Cell 25.8 - Draw chromosome-position summary of the locus distribution
# X = MG1655 chromosomal location.
# Y = mean Jaccard dissimilarity across the 48 matched pairs.
# Marker size = fraction of the 16 high-MIC pathogens whose exact state is
# absent from all 3 matched comparators.

x=locus_summary['reference_midpoint_Mb'].to_numpy(float)
yplot=locus_summary['mean_jaccard_dissimilarity_across_48_matched_pairs'].to_numpy(float)
frac=locus_summary['fraction_of_16_high_MIC_pathogens_with_exact_state_absent_from_all_3_comparators'].to_numpy(float)
sizes=40+360*frac
fig,ax=plt.subplots(figsize=(14,6))
ax.scatter(x,yplot,s=sizes,alpha=0.65)
for _,row in locus_summary.iterrows():
    ax.annotate(str(row['subwindow_name']),(float(row['reference_midpoint_Mb']),float(row['mean_jaccard_dissimilarity_across_48_matched_pairs'])),xytext=(0,6),textcoords='offset points',ha='center',rotation=90,fontsize=8)
ax.set_xlim(0,4.641652); ax.set_xlabel('MG1655 chromosomal location (Mb)')
ax.set_ylabel('Mean Jaccard dissimilarity across 48 matched pairs')
ax.set_title('Distribution of supported chromosomal differences in the 16 high-MIC pathogens')
handles=[]
for f in [0.25,0.50,0.75,1.0]:
    handles.append(ax.scatter([],[],s=40+360*f,alpha=0.65,label=f'{int(f*16)}/16'))
ax.legend(handles=handles,title='High-MIC pathogens whose exact state\nis absent from all 3 matched comparators',loc='upper left',bbox_to_anchor=(1.01,1.0))
fig.tight_layout(); fig.savefig(FIG25D,dpi=300,bbox_inches='tight'); fig.savefig(FIG25D.with_suffix('.pdf'),bbox_inches='tight'); plt.show(); plt.close(fig)
print('Saved:',FIG25D)
print('\nLocus summary ranked by mean matched-pair dissimilarity:')
display(locus_summary.sort_values('mean_jaccard_dissimilarity_across_48_matched_pairs',ascending=False).reset_index(drop=True))
print('Transition: Cell 25.9 will perform final QC and freeze the analysis.')


In [ ]:
#@title Cell 25.9 - Final QC and freeze the last analysis
# Notebook 25 is the final planned computational analysis before manuscript writing.

required=[PAIR_RESULTS,HIGH_RESULTS,LOCUS_SUMMARY,FIG25A,FIG25B,FIG25C,FIG25D,
          FIG25A.with_suffix('.pdf'),FIG25B.with_suffix('.pdf'),FIG25C.with_suffix('.pdf'),FIG25D.with_suffix('.pdf')]
for p in required:
    assert p.exists() and p.stat().st_size>0, f'Missing or empty output: {p}'
assert len(pair_results)==EXPECTED_PAIRS*EXPECTED_REGIONS
assert len(high_results)==EXPECTED_HIGH*EXPECTED_REGIONS
assert len(locus_summary)==EXPECTED_REGIONS
assert high_mean.shape==(EXPECTED_HIGH,EXPECTED_REGIONS)
assert high_absent.shape==(EXPECTED_HIGH,EXPECTED_REGIONS)
assert pair_matrix.shape==(EXPECTED_PAIRS,EXPECTED_REGIONS)
qc=pd.DataFrame([{'pathogens':EXPECTED_PATHOGENS,'high_MIC_pathogens':EXPECTED_HIGH,'matched_pairs':EXPECTED_PAIRS,
                  'unique_matched_comparators':EXPECTED_COMPARATORS,'comparators_per_high_MIC_pathogen':3,
                  'supported_mapped_loci':EXPECTED_REGIONS,'pair_region_rows':len(pair_results),
                  'high_pathogen_region_rows':len(high_results),'independent_confirmation_test':False,
                  'analysis_role':'secondary descriptive locus distribution','final_QC_pass':True}])
qc.to_csv(FINAL_QC,index=False)
COMPLETE.write_text(json.dumps({'status':'complete','analysis_role':'secondary descriptive locus distribution',
                                'high_MIC_pathogens':EXPECTED_HIGH,'matched_pairs':EXPECTED_PAIRS,
                                'supported_mapped_loci':mapped['subwindow_name'].astype(str).tolist(),
                                'independent_confirmation_test':False,'final_QC_pass':True},indent=2),encoding='utf-8')
print('Final QC: PASS'); display(qc)
print('\nNotebook 25 ends here. Review Figures 25A-25D, especially Figure 25D, then freeze the analysis for manuscript writing.')
